# Azure Databricks – PySpark Demo

This notebook demonstrates core Databricks / PySpark patterns:
- Creating a Spark session (auto-available in Databricks as `spark`)
- Reading CSV data
- Transformations: filter, groupBy, window functions, joins
- Writing to Delta Lake format
- Querying with Spark SQL

> **Note:** Run this notebook in an Azure Databricks workspace or locally with `pyspark` installed.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# In Databricks this is pre-created; locally we create it here
spark = (
    SparkSession.builder
    .appName("DatabricksPySparkDemo")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Read Sample Sales Data

In [ ]:
# In Databricks: spark.read.csv('/mnt/adls/raw/sales/')
# Locally: read the sample CSV provided in the adf module
import os
csv_path = os.path.join(os.path.dirname(os.getcwd()), 'adf', 'sample_sales.csv')

raw_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(csv_path)
)
raw_df.printSchema()
raw_df.show(truncate=False)

## 2. Clean & Transform

In [ ]:
clean_df = (
    raw_df
    .dropna(subset=['order_id', 'quantity', 'unit_price'])
    .filter(F.col('quantity').cast('double').isNotNull())
    .withColumn('quantity', F.col('quantity').cast('integer'))
    .withColumn('unit_price', F.col('unit_price').cast('double'))
    .withColumn('total', F.round(F.col('quantity') * F.col('unit_price'), 2))
    .withColumn('order_date', F.to_date('order_date', 'yyyy-MM-dd'))
)
print(f'Rows after cleaning: {clean_df.count()}')
clean_df.show(truncate=False)

## 3. Aggregations – Revenue by Product

In [ ]:
revenue_df = (
    clean_df
    .groupBy('product')
    .agg(
        F.count('order_id').alias('num_orders'),
        F.sum('total').alias('total_revenue'),
        F.avg('unit_price').alias('avg_unit_price'),
    )
    .orderBy(F.desc('total_revenue'))
)
revenue_df.show()

## 4. Window Function – Running Total per Customer

In [ ]:
window_spec = Window.partitionBy('customer_id').orderBy('order_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)

windowed_df = clean_df.withColumn('running_total', F.sum('total').over(window_spec))
windowed_df.select('order_id', 'customer_id', 'order_date', 'total', 'running_total').show()

## 5. Spark SQL

In [ ]:
clean_df.createOrReplaceTempView('orders')

spark.sql("""
    SELECT
        product,
        COUNT(*) AS orders,
        ROUND(SUM(total), 2) AS revenue
    FROM orders
    GROUP BY product
    ORDER BY revenue DESC
""").show()

## 6. Write to Delta Lake

In [ ]:
import tempfile, os
delta_path = os.path.join(tempfile.gettempdir(), 'databricks_demo_orders_delta')

# In Databricks: write to /mnt/adls/silver/orders/
(
    clean_df.write
    .format('delta')
    .mode('overwrite')
    .save(delta_path)
)
print(f'Delta table written to: {delta_path}')

# Read back
spark.read.format('delta').load(delta_path).show()

## Summary

| Step | PySpark API Used |
|---|---|
| Read CSV | `spark.read.csv()` |
| Clean / cast | `dropna`, `withColumn`, `cast` |
| Aggregate | `groupBy`, `agg` |
| Window function | `Window`, `sum().over()` |
| SQL | `createOrReplaceTempView`, `spark.sql()` |
| Delta write | `write.format('delta').save()` |